# AML Network Analysis: Geldwäscheringe mit NetworkX erkennen

**Lernziel:** Sie erstellen und analysieren Transaktionsnetzwerke:
- Aufbau eines gerichteten Graphen (Knoten = Konten, Kanten = Transaktionen)
- Zentralitätsmaße (Degree, Betweenness, PageRank)
- Zykluserkennung (Hinweis auf Smurfing)
- Community Detection (mögliche kriminelle Gruppen)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Gerichteter Graph (Knoten = Konten, Kanten = Transaktionen)
G = nx.DiGraph()

# Transaktionen: (sender, receiver, amount)
transactions = [
    ('A', 'B', 5000), ('B', 'C', 5000), ('C', 'A', 5000),  # Zyklus A->B->C->A
    ('A', 'D', 2000), ('D', 'E', 2000), ('E', 'F', 2000),
    ('X', 'Y', 10000), ('Y', 'Z', 10000),
    ('A', 'X', 3000)
]

for s, r, amt in transactions:
    G.add_edge(s, r, weight=amt)

print(f"Anzahl Konten: {G.number_of_nodes()}")
print(f"Anzahl Transaktionen: {G.number_of_edges()}")

# Visualisierung
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=500)
labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
plt.title('Transaktionsnetzwerk')
plt.show()

## 1. Zykluserkennung (Smurfing)

In [ ]:
cycles = list(nx.simple_cycles(G))
print(f"Gefundene Zyklen: {cycles}")
if cycles:
    print("⚠️ Zyklische Transaktionen entdeckt – möglicher Smurfing-Indikator.")

## 2. Zentralitätsmaße (wichtige Knoten)

In [ ]:
degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)
pagerank = nx.pagerank(G)

print("Degree Centrality:", sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True))
print("\nBetweenness Centrality:", sorted(betweenness.items(), key=lambda x: x[1], reverse=True))
print("\nPageRank:", sorted(pagerank.items(), key=lambda x: x[1], reverse=True))

## 3. Community Detection (Louvain)

In [ ]:
try:
    import community as community_louvain
    # Für ungerichteten Graphen (Kopie)
    G_undirected = G.to_undirected()
    partition = community_louvain.best_partition(G_undirected)
    print("Gemeinschaften:", set(partition.values()))
    nx.draw(G_undirected, pos, node_color=list(partition.values()), cmap=plt.cm.tab10, with_labels=True)
    plt.title('Community Detection')
    plt.show()
except ImportError:
    print("python-louvain nicht installiert. Führen Sie 'pip install python-louvain' aus.")

## Diskussion

- **Zyklen** deuten auf Smurfing hin (Geld wird im Kreis verschoben).
- **PageRank** identifiziert zentrale Konten (potenzielle Drahtzieher).
- **Community Detection** gruppiert Konten, die häufig miteinander transagieren – ideal für die Aufdeckung von Geldwäscheringen.
- In der Praxis würde man tausende Knoten analysieren und die Algorithmen auf echten Transaktionsdaten ausführen.